# Pipeline de Preprocesamiento de Microdatos: EAIMCS 2017-2018 (INE Bolivia)

> **Cuaderno Interactivo de Ingeniería de Datos y Preparación de Microdatos para Aprendizaje Supervisado (Regresión)**  
> **Estudio Oficial:** Encuesta a la Industria Manufacturera, Comercio y Servicios (EAIMCS 2017-2018)  
> **Catálogo ANDA (INE Bolivia):** [BOL-INE-EAIMCS-2017-2018](https://anda.ine.gob.bo/index.php/catalog/252)  
> **Objetivo del Proyecto:** Estimación y auditoría de los Ingresos Operativos Anuales (`S00_01_A` en Bolivianos) de medianas y grandes empresas en Bolivia.

---

### Propósito y Objetivos de este Cuaderno
Este notebook reproduce y explica exhaustivamente **todo el pipeline de preprocesamiento** aplicado a los microdatos brutos del INE para generar el dataset analítico final (`data/processed/dataset_procesado.csv` de **3.153 filas × 184 columnas**).

El flujo resuelve de manera sistemática los 5 grandes desafíos del dataset original:
1. **Unificación Relacional:** Fusión de tablas con distinta cardinalidad (módulo general $1:1$ vs. módulo de insumos $N:1$).
2. **Depuración Contable:** Saneamiento de tipos, importes negativos anómalos y auditoría de códigos centinela (`99999`).
3. **Agrupación Categórica:** Mapeo de más de 400 clases CAEB a 13 macrosectores económicos homogéneos según la CIIU Rev. 4.
4. **Prevención Estricta de Fuga de Información (*Anti-Leakage*):** Exclusión fundamentada de 19 variables que reproducen trivialmente el target o corresponden a agregados macroeconómicos calculados post-encuesta por el INE (`VBP`, `VA`, `CI`, `EAC`, `OGO`, etc.).
5. **Estabilización de Varianza:** Transformación logarítmica monótona $\text{log1p}(x) = \ln(1 + x)$ para corregir la severa asimetría de Pareto, con imputación de valores ausentes desacoplada y diferida al pipeline de modelado.


## 1. Configuración del Entorno y Librerías

Importamos las librerías necesarias para el procesamiento relacional, álgebra lineal, precisión numérica decimal y visualización interactiva.


In [1]:
import os
import sys
import json
import hashlib
from decimal import Decimal, InvalidOperation
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configurar visualización en pandas
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Resolver rutas relativas al proyecto
current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Directorio raíz del proyecto:", PROJECT_ROOT)
print("Directorio de datos crudos:  ", RAW_DIR)
print("Directorio procesado:        ", PROCESSED_DIR)


Directorio raíz del proyecto: C:\Users\raque\OneDrive\Documentos\ProyML\finalProjectML
Directorio de datos crudos:   C:\Users\raque\OneDrive\Documentos\ProyML\finalProjectML\data\raw
Directorio procesado:         C:\Users\raque\OneDrive\Documentos\ProyML\finalProjectML\data\processed


## 2. Carga y Verificación Criptográfica de los Datos Crudos

La encuesta se distribuye en múltiples archivos de microdatos. Para el modelo utilizaremos los dos archivos clave:
1. **`MOD_ANUAL_S01-07_12_general_i.csv` (Módulo General):** Contiene la carátula de identificación de la empresa, ingresos globales, personal ocupado, masa salarial, servicios básicos, inventarios y activos fijos.
2. **`MOD_ANUAL_S10_materiales_i.csv` (Sección 10 - Materias Primas):** Detalle en estructura larga ($N:1$) de materias primas, insumos y envases consumidos en procesos fabriles.

Para garantizar la reproducibilidad y verificar que los archivos de entrada no han sido alterados, calculamos su hash criptográfico SHA-256.


In [2]:
def compute_sha256(file_path: Path) -> str:
    digest = hashlib.sha256()
    with file_path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

path_general = RAW_DIR / "MOD_ANUAL_S01-07_12_general_i.csv"
path_materiales = RAW_DIR / "MOD_ANUAL_S10_materiales_i.csv"

hash_gen = compute_sha256(path_general)
hash_mat = compute_sha256(path_materiales)

print(f"SHA-256 General:    {hash_gen}")
print(f"SHA-256 Materiales: {hash_mat}")


SHA-256 General:    e711b839d1e83e62b7e81ec64d0f64ea7a2d0eb06974662e99c5bfdc204927f7
SHA-256 Materiales: b41ad4ed15c427cb20c25a76732ff43f47834158b7eadf6c58a5141b80e18b4a


### Carga de DataFrames y Tratamiento de Filas Físicas Vacías

Cargamos los microdatos. En el archivo de materiales, el CSV crudo contiene una fila física completamente en blanco (`NaN` en todas sus celdas) producto del volcado de exportación SPSS original. La detectamos y filtramos antes de validar la integridad de claves.


In [3]:
# Cargar tabla general de empresas
df_gen_raw = pd.read_csv(
    path_general, 
    low_memory=False, 
    dtype={"S00_01_A": "string", "S05_04": "string"}
)

# Cargar tabla de insumos y materiales
df_mat_raw = pd.read_csv(path_materiales, low_memory=False)
filas_mat_inicial = len(df_mat_raw)

# Detectar y excluir filas completamente vacías
blank_rows = df_mat_raw.isna().all(axis=1)
df_mat_clean = df_mat_raw.loc[~blank_rows].copy()
filas_blanco_excluidas = int(blank_rows.sum())

print(f"Módulo General:    {df_gen_raw.shape[0]:,} empresas × {df_gen_raw.shape[1]} columnas")
print(f"Módulo Materiales: {filas_mat_inicial:,} filas físicas -> {len(df_mat_clean):,} registros válidos (excluidas {filas_blanco_excluidas} fila en blanco)")
print(f"Empresas únicas en General:    {df_gen_raw['ID'].nunique():,} (sin duplicados: {df_gen_raw['ID'].is_unique})")
print(f"Empresas únicas en Materiales: {df_mat_clean['ID'].nunique():,}")


Módulo General:    3,153 empresas × 167 columnas
Módulo Materiales: 6,428 filas físicas -> 6,427 registros válidos (excluidas 1 fila en blanco)
Empresas únicas en General:    3,153 (sin duplicados: True)
Empresas únicas en Materiales: 1,614


In [4]:
# Vista previa de las primeras observaciones del Módulo General
display(df_gen_raw[["ID", "C2_01", "actividad_pricipal_codigo_V1", "S00_01_A", "S01_05_A", "S01_03_C", "S02_09"]].head())


          ID       C2_01  actividad_pricipal_codigo_V1            S00_01_A  \
0 296,307.00  SANTA CRUZ                         10104        1112639496.0   
1 538,207.00  SANTA CRUZ                          1010         222819434.0   
2 544,316.00  SANTA CRUZ                         10101         152951332.0   
3 547,760.00  COCHABAMBA                         10104  191899185.55228135   
4 271,912.00      LA PAZ                         10106          94239676.0   

   S01_05_A       S01_03_C       S02_09  
0  2,916.00 106,686,129.00 7,645,956.00  
1    530.00  21,144,426.00 4,159,458.00  
2    300.00  16,260,779.00 2,579,961.00  
3    856.00  23,380,892.00 3,800,193.00  
4    273.00  18,977,988.00 1,478,499.00  


In [5]:
# Vista previa del Módulo de Materiales (estructura larga)
display(df_mat_clean.head(8))


         ID                                   materia  codigocodif_V1  \
0  5,986.00                   MATERIALES (INVENTARIO)   99,999,994.00   
1 10,824.00                                    VARIOS   99,999,994.00   
2 16,646.00                             PIEDRA CALIZA   15,120,003.00   
3 16,769.00                                   CEMENTO   37,440,001.00   
4 16,851.00                            INSUMOS VARIOS   99,999,994.00   
5 20,746.00    POSTES DE EUCALIPTO DIFERENTES TAMAÑOS   31,330,002.00   
6 20,746.00  TUBERIAS PVC SDR 41 DIFERENTES DIAMETROS   36,320,001.00   
7 20,746.00                                   CEMENTO   37,440,001.00   

             unidad  cantidad_co     valor_co  cantidad_uti    valor_uti  
0             VALOR         0.00 1,669,035.00          0.00 1,557,660.00  
1             VALOR         0.00 4,542,856.92          0.00 3,384,641.92  
2  TONELADA MÉTRICA    23,186.89 1,051,526.00     41,873.00 2,701,859.65  
3         KILOGRAMO 2,820,742.00 2,401,832

## 3. Etapa 1: Validación y Depuración de Valores Monetarios

### Marco Metodológico: Códigos Centinela (`99999`) y Valores Negativos
En las encuestas económicas del INE, se contempló el uso del código **`99999`** en reglas específicas de balance de materias primas:
$$\text{Utilización} = \text{Compras} + \text{Inventario Inicial} - \text{Inventario Final}$$
Cuando el balance no cuadraba en la boleta web o persistían inconsistencias de autorelevamiento, se asignaba dicho centinela.

**Regla de Oro del Preprocesamiento:**
* Un código centinela sólo se recodifica como nulo si se documenta formalmente su uso en la variable analizada.
* Un valor numérico elevado **no es un valor perdido**: las grandes empresas manufactureras en Bolivia manejan compras de cientos de millones de Bolivianos.
* Los importes monetarios deben ser estrictamente no negativos ($\ge 0$). Valores negativos indican errores de digitación y se transforman a `NaN` (no a cero artificial).


In [6]:
# Auditoría del centinela 99999 en materiales y predictores
CENTINELA_VAL = 99999.0

centinelas_mat = int((df_mat_clean[["valor_co", "valor_uti"]].isin([CENTINELA_VAL, str(int(CENTINELA_VAL))])).sum().sum())
print(f"Ocurrencias exactas de 99999 en valor_co y valor_uti de materiales: {centinelas_mat}")

# Coerción numérica y tratamiento de importes negativos
for col in ["valor_co", "valor_uti"]:
    s = pd.to_numeric(df_mat_clean[col], errors="coerce")
    negativos = (s < 0).sum()
    print(f"Columna '{col}': {negativos} valores negativos detectados.")
    df_mat_clean[f"{col}_clean"] = s.where(s >= 0)

print("\nDepuración completada. Los negativos se reemplazaron por NaN (sin alterar ceros legítimos ni montos elevados).")


Ocurrencias exactas de 99999 en valor_co y valor_uti de materiales: 0
Columna 'valor_co': 1 valores negativos detectados.
Columna 'valor_uti': 0 valores negativos detectados.

Depuración completada. Los negativos se reemplazaron por NaN (sin alterar ceros legítimos ni montos elevados).


## 4. Etapa 2: Agregación Relacional de la Sección 10 (Materias Primas)

La tabla de materiales tiene **6.427 registros** pertenecientes a **1.614 empresas únicas** (exclusivamente aquellas que reportaron actividad fabril o manufacturera).

Para incorporarla a nuestro modelo a nivel de empresa, debemos colapsar la granularidad mediante un `groupby('ID')`:
1. **`n_insumos`:** Variedad de insumos declarados por la empresa:
   $$\text{n\_insumos} = \text{count}(\text{materia})$$
2. **`total_valor_co`:** Monto total de compras de materias primas en Bolivianos:
   $$\text{total\_valor\_co} = \sum \text{valor\_co\_clean}$$
3. **`total_valor_uti`:** Consumo real de materias primas e insumos en el proceso productivo:
   $$\text{total\_valor\_uti} = \sum \text{valor\_uti\_clean}$$

> **Detalle técnico de integridad:** Se utiliza `min_count=1` en la sumatoria para garantizar que si una empresa reportó filas de insumos pero todos sus valores monetarios eran ausentes (`NaN`), la suma resultante sea `NaN` y no un falso `0.0`.


In [7]:
df_mat_agg = df_mat_clean.groupby("ID").agg(
    n_insumos=("ID", "size"),
    total_valor_co=("valor_co_clean", lambda s: s.sum(min_count=1)),
    total_valor_uti=("valor_uti_clean", lambda s: s.sum(min_count=1))
).reset_index()

print(f"Empresas manufactureras agregadas: {len(df_mat_agg):,}")
display(df_mat_agg.describe())


Empresas manufactureras agregadas: 1,614
                 ID  n_insumos   total_valor_co  total_valor_uti
count      1,614.00   1,614.00         1,613.00         1,614.00
mean   5,597,899.03       3.98    21,669,460.79    21,527,083.91
std    7,619,885.49       5.78   128,894,657.96   127,682,602.54
min        5,986.00       1.00             0.00             0.00
25%      525,681.50       1.00       461,473.00       628,285.86
50%    4,493,210.50       1.00     3,230,470.31     3,204,284.00
75%    7,058,539.50       5.00     9,199,926.00     9,141,168.99
max   40,971,628.00      67.00 3,867,070,672.42 3,838,233,308.47


In [8]:
# Gráfico interactivo: Variedad de materias primas declaradas por empresa
fig_insumos = px.histogram(
    df_mat_agg, 
    x="n_insumos", 
    nbins=25,
    title="Distribución de la Variedad de Insumos Declarados (Sección 10)",
    labels={"n_insumos": "Cantidad de Insumos Declarados", "count": "N° de Empresas"},
    color_discrete_sequence=["#1f77b4"]
)
fig_insumos.update_layout(template="plotly_white", bargap=0.1)
fig_insumos.show()


[Gráfico Interactivo Plotly: 'Distribución de la Variedad de Insumos Declarados (Sección 10)']


## 5. Etapa 3: Fusión Relacional (Left Join) y Normalización Categórica

Unimos la tabla general de 3.153 empresas con la tabla agregada de insumos mediante un `Left Join` por la clave `ID`.

### Semántica de las Empresas sin Insumos
* **`n_insumos`:** Las 1.539 empresas comerciales y de servicios que legítimamente no consumen materias primas de transformación reciben `n_insumos = 0` (cero insumos declarados).
* **`total_valor_co` y `total_valor_uti`:** **Permanecen como ausentes (`NaN`)**. Fabricar un gasto de Bs 0 distorsionaría la interpretación económica, pues no es que hayan comprado Bs 0 de materias primas, sino que la sección fabril no les corresponde. La imputación estadística se difiere al pipeline de Machine Learning.


In [9]:
# Validación estricta de cardinalidad uno a uno
merged_df = df_gen_raw.merge(df_mat_agg, on="ID", how="left", validate="one_to_one")

# Bandera de pertenencia al módulo fabril
merged_df["has_material_record"] = merged_df["n_insumos"].notna()

# Imputación semántica de n_insumos
merged_df["n_insumos"] = merged_df["n_insumos"].fillna(0)

# Verificación de ausentes monetarios conservados
print(f"Empresas con registro en Sección 10: {merged_df['has_material_record'].sum():,}")
print(f"Empresas sin registro en Sección 10: {(~merged_df['has_material_record']).sum():,}")
print(f"Faltantes en total_valor_co:          {merged_df['total_valor_co'].isna().sum():,}")
print(f"Faltantes en total_valor_uti:         {merged_df['total_valor_uti'].isna().sum():,}")


Empresas con registro en Sección 10: 1,614
Empresas sin registro en Sección 10: 1,539
Faltantes en total_valor_co:          1,540
Faltantes en total_valor_uti:         1,539


### Normalización Territorial (`depto`)
Estandarizamos el campo de departamento `C2_01`, eliminando espacios en blanco y convirtiendo cadenas a mayúsculas estrictas.


In [10]:
merged_df["depto"] = merged_df["C2_01"].astype(str).str.strip().str.upper()

dist_depto = merged_df["depto"].value_counts().reset_index()
dist_depto.columns = ["Departamento", "Empresas"]
dist_depto["Porcentaje"] = (dist_depto["Empresas"] / len(merged_df) * 100).round(2)

fig_depto = px.bar(
    dist_depto,
    x="Departamento",
    y="Empresas",
    text="Empresas",
    title="Distribución de Empresas por Departamento (EAIMCS 2017-2018)",
    color="Empresas",
    color_continuous_scale="Blues"
)
fig_depto.update_traces(textposition="outside")
fig_depto.update_layout(template="plotly_white")
fig_depto.show()

display(dist_depto)


[Gráfico Interactivo Plotly: 'Distribución de Empresas por Departamento (EAIMCS 2017-2018)']
  Departamento  Empresas  Porcentaje
0   SANTA CRUZ      1427       45.26
1   COCHABAMBA       649       20.58
2       LA PAZ       631       20.01
3       TARIJA       131        4.15
4        ORURO       110        3.49
5   CHUQUISACA        74        2.35
6       POTOSI        58        1.84
7         BENI        50        1.59
8        PANDO        23        0.73


### Normalización de Actividades Económicas CAEB a Macrosectores
El clasificador CAEB a 5 dígitos presenta más de 400 clases, lo que provocaría dispersión dimensional inmanejable.

Utilizamos la función `map_caeb_to_sector` para agrupar los dos primeros dígitos de la actividad principal según las grandes divisiones de la **CIIU Rev. 4** en 14 macrosectores teóricos (de los cuales **13 se observan en la muestra local**).


In [11]:
def map_caeb_to_sector(code) -> str:
    try:
        c_str = str(code).strip()[:2]
        num = int(c_str)
        if 1 <= num <= 3:
            return "Agropecuario y Pesca"
        elif 5 <= num <= 9:
            return "Minería e Hidrocarburos"
        elif 10 <= num <= 33:
            return "Industria Manufacturera"
        elif 35 <= num <= 39:
            return "Electricidad, Gas y Agua"
        elif 41 <= num <= 43:
            return "Construcción"
        elif 45 <= num <= 47:
            return "Comercio Mayorista y Minorista"
        elif 49 <= num <= 53:
            return "Transporte y Almacenamiento"
        elif 55 <= num <= 56:
            return "Alojamiento y Servicios de Comida"
        elif 58 <= num <= 63:
            return "Información y Comunicaciones"
        elif 64 <= num <= 66:
            return "Intermediación Financiera"
        elif 68 <= num <= 68:
            return "Actividades Inmobiliarias"
        elif 69 <= num <= 75:
            return "Servicios Profesionales y Técnicos"
        elif 77 <= num <= 82:
            return "Servicios Administrativos y de Apoyo"
        elif 85 <= num <= 85:
            return "Educación"
        elif 86 <= num <= 88:
            return "Salud y Asistencia Social"
        else:
            return "Otras Actividades de Servicios"
    except Exception:
        return "Otras Actividades de Servicios"

merged_df["sector_macro"] = merged_df["actividad_pricipal_codigo_V1"].apply(map_caeb_to_sector)

dist_sector = merged_df["sector_macro"].value_counts().reset_index()
dist_sector.columns = ["Macrosector", "Empresas"]
dist_sector["Porcentaje"] = (dist_sector["Empresas"] / len(merged_df) * 100).round(2)

fig_sector = px.bar(
    dist_sector,
    x="Empresas",
    y="Macrosector",
    orientation="h",
    text="Empresas",
    title="Distribución de Empresas por Macrosector Económico (13 Sectores Observados)",
    color="Empresas",
    color_continuous_scale="Viridis"
)
fig_sector.update_traces(textposition="outside")
fig_sector.update_layout(template="plotly_white", yaxis={'categoryorder': 'total ascending'})
fig_sector.show()

display(dist_sector)


[Gráfico Interactivo Plotly: 'Distribución de Empresas por Macrosector Económico (13 Sectores Observados)']
                             Macrosector  Empresas  Porcentaje
0         Comercio Mayorista y Minorista      1243       39.42
1                Industria Manufacturera       489       15.51
2                           Construcción       430       13.64
3            Transporte y Almacenamiento       182        5.77
4      Alojamiento y Servicios de Comida       137        4.35
5     Servicios Profesionales y Técnicos       135        4.28
6   Servicios Administrativos y de Apoyo       122        3.87
7           Información y Comunicaciones       114        3.62
8                              Educación        94        2.98
9              Salud y Asistencia Social        77        2.44
10              Electricidad, Gas y Agua        54        1.71
11             Actividades Inmobiliarias        40        1.27
12        Otras Actividades de Servicios        36        1.14


## 6. Etapa 4: Políticas Anti-Leakage y Conciliación del Objetivo

### 6.1 Conciliación de la Variable Objetivo: `S00_01_A` vs `S05_04`
En la encuesta oficial, los ingresos operativos anuales aparecen en dos campos:
1. `S00_01_A`: Valor total de ingresos operativos (Carátula contable).
2. `S05_04`: Suma de ventas de manufactura (`S05_01`), ventas de mercadería (`S05_02`) y servicios (`S05_03`).

Realizamos una conciliación aritmética de alta precisión utilizando `Decimal`:


In [12]:
def target_differences(df: pd.DataFrame) -> pd.Series:
    def diff_row(row):
        left, right = row["S00_01_A"], row["S05_04"]
        if pd.isna(left) or pd.isna(right):
            return np.nan
        try:
            return float(abs(Decimal(str(left)) - Decimal(str(right))))
        except InvalidOperation:
            return np.nan
    return df[["S00_01_A", "S05_04"]].apply(diff_row, axis=1)

diferencias = target_differences(merged_df)
coincidentes_exactos = (diferencias == 0).sum()
diferencias_redondeo = ((diferencias > 0) & (diferencias <= 1.0)).sum()
discrepancias_mayores = (diferencias > 1.0).sum()

print("=== REPORTE DE CONCILIACIÓN DE LA VARIABLE OBJETIVO ===")
print(f"Coincidencias idénticas exactas:              {coincidentes_exactos:,} ({coincidentes_exactos/len(merged_df)*100:.2f} %)")
print(f"Diferencias menores por redondeo (<= Bs 1.0): {diferencias_redondeo:,} ({diferencias_redondeo/len(merged_df)*100:.2f} %)")
print(f"Discrepancias mayores a Bs 1.0:                {discrepancias_mayores} (0.00 %)")
print(f"Diferencia máxima encontrada:                 Bs {diferencias.max():.2f}")

# Fijar formalmente la variable objetivo
merged_df["target"] = pd.to_numeric(merged_df["S00_01_A"], errors="coerce")

# Validar filtro de ingresos positivos
valid_mask = merged_df["target"].notnull() & (merged_df["target"] > 0)
print(f"Total empresas con target positivo válido:    {valid_mask.sum():,} de {len(merged_df):,} (100% retenidas)")


=== REPORTE DE CONCILIACIÓN DE LA VARIABLE OBJETIVO ===
Coincidencias idénticas exactas:              3,093 (98.10 %)
Diferencias menores por redondeo (<= Bs 1.0): 60 (1.90 %)
Discrepancias mayores a Bs 1.0:                0 (0.00 %)
Diferencia máxima encontrada:                 Bs 1.00
Total empresas con target positivo válido:    3,153 de 3,153 (100% retenidas)


### 6.2 Exclusión Estricta de 19 Variables de Fuga de Datos (*Data Leakage*)

Para que un modelo de Machine Learning aprenda a estimar la facturación a partir de la capacidad productiva instalada y no memorice identidades contables triviales, debemos aislar del entrenamiento:
1. **Componentes de Ingresos de Sección 5:** `S05_01`, `S05_02`, `S05_03`, `S05_04` (su suma es igual al target).
2. **Cuentas de Producción de Cuentas Nacionales del INE:** `VPA`, `PC`, `ISPOINF`, `VIPP`, `VBP` (Valor Bruto de Producción, que incluye directamente ventas ajustadas).
3. **Cuentas de Valor Agregado:** `EAC`, `OGO` (Excedente Bruto), `VUMPEEI`, `CI` (Consumo Intermedio), `VA` (Valor Agregado Bruto: $VBP - CI$).
4. **Ratios e Indicadores Post-Encuesta del INE:** `SSB`, `OPP`, `PS`, `R`, `D`.


In [13]:
EXCLUDED_LEAKAGE_COLS = [
    "S05_01", "S05_02", "S05_03", "S05_04",
    "VPA", "PC", "ISPOINF", "VIPP", "VBP",
    "EAC", "OGO", "VUMPEEI", "CI", "VA",
    "SSB", "OPP", "PS", "R", "D"
]

presentes_fuga = [c for c in EXCLUDED_LEAKAGE_COLS if c in merged_df.columns]
print(f"Variables identificadas como FUGA DE DATOS presentes en el dataset: {len(presentes_fuga)} de 19")
for i, col in enumerate(presentes_fuga, 1):
    print(f" {i:2d}. {col:10s} -> Excluida del espacio predictor")


Variables identificadas como FUGA DE DATOS presentes en el dataset: 19 de 19
  1. S05_01     -> Excluida del espacio predictor
  2. S05_02     -> Excluida del espacio predictor
  3. S05_03     -> Excluida del espacio predictor
  4. S05_04     -> Excluida del espacio predictor
  5. VPA        -> Excluida del espacio predictor
  6. PC         -> Excluida del espacio predictor
  7. ISPOINF    -> Excluida del espacio predictor
  8. VIPP       -> Excluida del espacio predictor
  9. VBP        -> Excluida del espacio predictor
 10. EAC        -> Excluida del espacio predictor
 11. OGO        -> Excluida del espacio predictor
 12. VUMPEEI    -> Excluida del espacio predictor
 13. CI         -> Excluida del espacio predictor
 14. VA         -> Excluida del espacio predictor
 15. SSB        -> Excluida del espacio predictor
 16. OPP        -> Excluida del espacio predictor
 17. PS         -> Excluida del espacio predictor
 18. R          -> Excluida del espacio predictor
 19. D          -> Excl

### 6.3 Política sobre Variables de Capacidad de Almacenamiento (Sección 12)
Las variables `S12_01_B` (capacidad máxima de materia prima) y `S12_02_B` (capacidad de producto) registran cantidades físicas numéricas, pero sus unidades de medida asociadas (`S12_01_C` y `S12_02_C`) corresponden a unidades heterogéneas (litros, toneladas, unidades, metros cúbicos, cajas).

Sumar o comparar estas cifras sin estandarización física previa introduciría ruido espurio. Por tanto, **se excluyen de la lista de predictores del modelo**, conservándose únicamente como variables descriptivas para auditoría.


## 7. Etapa 5: Transformaciones Logarítmicas de Estabilización de Varianza

### Justificación Matemática de $\text{log1p}(x) = \ln(1 + x)$
Las magnitudes monetarias en encuestas a empresas medianas y grandes exhiben una **distribución de Pareto con colas pesadas**: la mayoría de empresas factura decenas de millones mientras que unas pocas alcanzan miles de millones.

Aplicamos la función monótona:
$$x_{\log} = \ln(1 + x) = \text{log1p}(x)$$
Ventajas clave:
1. Permite incorporar el cero sin generar indeterminaciones matemáticas: $\ln(1 + 0) = 0$.
2. Comprime órdenes de magnitud facilitando la convergencia de modelos lineales y ensambles de árboles.
3. Transforma relaciones de producción multiplicativas (tipo Cobb-Douglas: $Y = A K^\alpha L^\beta$) en formas lineales aditivas.


In [14]:
# Comparación visual del Target antes y después de log1p
fig_target = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("Escala Original (Bolivianos - Asimetría Severa)", "Escala Logarítmica log1p(Target) - Simétrica")
)

fig_target.add_trace(
    go.Histogram(x=merged_df["target"], nbinsx=50, marker_color="#d62728", name="Bs"),
    row=1, col=1
)
fig_target.add_trace(
    go.Histogram(x=np.log1p(merged_df["target"]), nbinsx=40, marker_color="#2ca02c", name="log(Bs)"),
    row=1, col=2
)
fig_target.update_layout(title_text="Efecto de la Estabilización de Varianza en el Ingreso Operativo", template="plotly_white", showlegend=False)
fig_target.show()


[Gráfico Interactivo Plotly: 'Efecto de la Estabilización de Varianza en el Ingreso Operativo']


### Aplicación de $\text{log1p}$ al Target y a los 9 Predictores Numéricos Clave


In [15]:
PREDICTOR_NUM_COLS = [
    "S01_05_A",         # Total personal ocupado
    "S01_03_C",         # Sueldos y salarios básicos anuales
    "S01_14",           # Otras remuneraciones y beneficios
    "S02_09",           # Total energía, agua y combustibles
    "S07_09_E",         # Activos fijos (valor histórico final)
    "S06_06_B",         # Total inventarios finales
    "n_insumos",        # Variedad de materias primas
    "total_valor_co",   # Compras de materias primas en Bs
    "total_valor_uti"   # Utilización de materias primas en Bs
]

# Crear copia procesada final
df_procesado = merged_df.copy()

# Target logarítmico
df_procesado["target_log"] = np.log1p(df_procesado["target"])

# Transformar los 9 predictores numéricos preservando NaN
for col in PREDICTOR_NUM_COLS:
    if col in df_procesado.columns:
        s = pd.to_numeric(df_procesado[col], errors="coerce")
        df_procesado[col] = s.where(s >= 0)
        df_procesado[f"log_{col}"] = np.log1p(df_procesado[col])

print(f"Transformación log1p aplicada exitosamente a {len(PREDICTOR_NUM_COLS)} predictores y al target.")


Transformación log1p aplicada exitosamente a 9 predictores y al target.


In [16]:
# Resumen de las 9 columnas predictoras transformadas
log_cols = [f"log_{c}" for c in PREDICTOR_NUM_COLS]
display(df_procesado[log_cols].describe())


       log_S01_05_A  log_S01_03_C  log_S01_14  log_S02_09  log_S07_09_E  \
count      3,153.00      3,153.00    3,153.00    3,153.00      3,153.00   
mean           3.55         14.13       13.28       11.70         15.42   
std            1.23          1.53        2.06        2.37          2.41   
min            0.69          0.00        0.00        0.00          0.00   
25%            2.64         13.14       12.21       10.60         14.21   
50%            3.35         14.00       13.30       11.79         15.56   
75%            4.28         15.04       14.46       13.03         16.73   
max            8.46         19.95       20.03       20.18         23.63   

       log_S06_06_B  log_n_insumos  log_total_valor_co  log_total_valor_uti  
count      3,153.00       3,153.00            1,613.00             1,614.00  
mean          10.77           0.64               13.73                14.36  
std            6.77           0.82                4.29                 2.89  
min         

In [17]:
# Matriz de correlación de Pearson entre los predictores logarítmicos y el target logarítmico
corr_matrix = df_procesado[["target_log"] + log_cols].corr()

fig_corr = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    title="Matriz de Correlación de Pearson: Predictores Logarítmicos vs. Target Logarítmico",
    aspect="auto"
)
fig_corr.update_layout(template="plotly_white")
fig_corr.show()


[Gráfico Interactivo Plotly: 'Matriz de Correlación de Pearson: Predictores Logarítmicos vs. Target Logarítmico']


## 8. Etapa 6: Verificación de Contrato de Calidad y Exportación

Verificamos formalmente las condiciones del contrato de datos:
* Conservación exacta de las 3.153 empresas sin pérdidas de filas.
* Cardinalidad de clave única `ID`.
* Presencia de las 184 columnas esperadas.
* 13 macrosectores observados.


In [18]:
# Aserciones de integridad y contrato de calidad
assert len(df_procesado) == 3153, f"Error en filas: esperado 3153, obtenido {len(df_procesado)}"
assert df_procesado["ID"].nunique() == 3153, "Error: Existen IDs duplicados"
assert df_procesado["sector_macro"].nunique() == 13, f"Error en macrosectores: esperado 13, obtenido {df_procesado['sector_macro'].nunique()}"
assert len(df_procesado.columns) == 184, f"Error en columnas: esperado 184, obtenido {len(df_procesado.columns)}"

print("¡TODAS LAS COMPROBACIONES DE CALIDAD DEL CONTRATO FUERON SATISFECHAS EXITOSAMENTE!")
print(f"Dimensiones finales del dataset: {df_procesado.shape[0]:,} filas × {df_procesado.shape[1]} columnas")


¡TODAS LAS COMPROBACIONES DE CALIDAD DEL CONTRATO FUERON SATISFECHAS EXITOSAMENTE!
Dimensiones finales del dataset: 3,153 filas × 184 columnas


In [19]:
# Exportar el dataset procesado canónico a data/processed/
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
csv_output_path = PROCESSED_DIR / "dataset_procesado.csv"

df_procesado.to_csv(csv_output_path, index=False, encoding="utf-8")
print(f"Dataset exportado exitosamente a: {csv_output_path}")

# Exportar a formato Parquet si pyarrow / fastparquet está disponible
try:
    parquet_path = PROCESSED_DIR / "dataset_procesado.parquet"
    df_procesado.to_parquet(parquet_path, index=False)
    print(f"Dataset exportado exitosamente a: {parquet_path}")
except Exception as e:
    print("Exportación a Parquet omitida (motor no instalado): salida CSV lista para consumo.")


Dataset exportado exitosamente a: C:\Users\raque\OneDrive\Documentos\ProyML\finalProjectML\data\processed\dataset_procesado.csv
Exportación a Parquet omitida (motor no instalado): salida CSV lista para consumo.


## 9. Conclusiones y Preparación para el Modelado Supervisado

El pipeline de preprocesamiento ha generado un dataset estructurado, libre de fugas de datos y con estabilidad de varianza garantizada:

| Componente | Estado Final |
|---|---|
| **Observaciones** | 3.153 empresas medianas y grandes (100% preservadas) |
| **Columnas Totales** | 184 columnas (167 base + 17 generadas) |
| **Variable Objetivo** | `target` (Bs) y `target_log` ($\ln(1 + \text{target})$) |
| **Predictores Nucleares** | 9 variables logarítmicas + 2 variables categóricas (`depto`, `sector_macro`) |
| **Variables Anti-Leakage** | 19 columnas de componentes de ingresos y cuentas de Cuentas Nacionales excluidas |
| **Imputación de Faltantes** | Preservados como `NaN` para ser imputados por mediana únicamente sobre los pliegues de entrenamiento de Scikit-Learn |

El dataset procesado está listo para ser utilizado por los algoritmos de regresión en `models/train.py` (Ridge, Random Forest, HistGradientBoosting).
